In [ ]:
import base64
from typing import cast

from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
 
# Load public key
with open("public_key.pem", "rb") as f:
    public_key = cast(rsa.RSAPublicKey, serialization.load_pem_public_key(f.read()))
 
password = ""
 
encrypted = public_key.encrypt(
    password.encode(),
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None
    )
)
 
encrypted_password = base64.b64encode(encrypted).decode()
 
# print(encrypted_password)

In [ ]:
import os
import requests

BASE = "https://analytics-dev.tatapower.com/TpcDSMPortal/api"

# Headers that make us look like a normal client (defeats WAF 406 blocks)
COMMON_HEADERS = {
    "User-Agent": "PostmanRuntime/7.39.0",   # or a browser UA string
    "Accept": "*/*",
}

# --- Step 1: get an access token ---
def get_token() -> str:
    resp = requests.post(
        f"{BASE}/token",
        headers=COMMON_HEADERS,
        data={  # form-urlencoded, matches Postman's "urlencoded" body
            "grant_type": "password",
            "username": "",
            "password": "",
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()["access_token"]  # adjust key if the response differs


# --- Step 2: call the utilityData endpoint ---
def get_utility_data(token: str, plant_name: str, date: str, schd_rev_no: int = -1) -> dict:
    resp = requests.post(
        f"{BASE}/utilityData",
        headers={**COMMON_HEADERS, "Authorization": f"Bearer {token}"},
        json={  # JSON body, matches Postman's "raw"/json body
            "plant_name": plant_name,
            "date": date,
            "schd_rev_no": schd_rev_no,
        },
        timeout=30,
    )
    resp.raise_for_status()
    return resp.json()


# --- Usage ---
# Read credentials from environment variables, NOT hardcoded in the notebook
token = get_token()
data = get_utility_data(token, plant_name="Pavagada", date="2026-06-03", schd_rev_no=-1)
print(data)


In [ ]:
data

### Explore json loaded from postgres


In [1]:

import psycopg2
import datetime
import os
import json

with open("local.settings.json", "r") as f:
    settings = json.load(f)["Values"]

for key, value in settings.items():
    os.environ[key] = value

database = os.getenv("POSTGRES_DB")
user = os.getenv("POSTGRES_USER")
password = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_HOST")

conn = psycopg2.connect(
    dbname=database,
    user=user,
    host=host,
    password=password,
    port="5432",
)

# 2026-06-19 01:30:00.739
right_time = datetime.datetime(2026, 6, 29, 7, 42, 0, 561776)

with conn.cursor() as cur:
    cur.execute(
        """
        SELECT
            schedule_date_ist,
            time_retrieved,
            time_retrieved_ist,
            schedule_data       
        FROM tata_schedule
        WHERE schedule_date_ist = %s
          AND time_retrieved_ist = %s;
        """,
        ("2026-06-29", right_time),
    )
    rows = cur.fetchall()

conn.close()
rows

[(datetime.date(2026, 6, 29),
  datetime.datetime(2026, 6, 29, 2, 12, 0, 561776, tzinfo=datetime.timezone.utc),
  datetime.datetime(2026, 6, 29, 7, 42, 0, 561776),
  {'data': {'ResponseBody': {'Date': '29-06-2026',
     'ApiRemarks': None,
     'ScheduleRemarks': 'FULLSCHEDULE_CREATED : From cron job',
     'GroupWiseDataList': [{'ASList': [],
       'Acronym': 'PVG_TATARENEWABLE',
       'MTDLRefList': [],
       'FullschdList': [{'ASTypeName': 'NA',
         'FullScheduleData': {'ASFullScheduleJsonData': None,
          'OAFullScheduleJsonData': {'SchdAmount': [0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0,
            0.06,
            1.82,
            4.01,
            6.7,
            9.71,
        

In [2]:
row = rows[0]


In [4]:
row

(datetime.date(2026, 6, 29),
 datetime.datetime(2026, 6, 29, 2, 12, 0, 561776, tzinfo=datetime.timezone.utc),
 datetime.datetime(2026, 6, 29, 7, 42, 0, 561776),
 {'data': {'ResponseBody': {'Date': '29-06-2026',
    'ApiRemarks': None,
    'ScheduleRemarks': 'FULLSCHEDULE_CREATED : From cron job',
    'GroupWiseDataList': [{'ASList': [],
      'Acronym': 'PVG_TATARENEWABLE',
      'MTDLRefList': [],
      'FullschdList': [{'ASTypeName': 'NA',
        'FullScheduleData': {'ASFullScheduleJsonData': None,
         'OAFullScheduleJsonData': {'SchdAmount': [0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0,
           0.06,
           1.82,
           4.01,
           6.7,
           9.71,
           12.91,
           15.36,
           

In [5]:
for item in row:
    print(item)

2026-06-29
2026-06-29 02:12:00.561776+00:00
2026-06-29 07:42:00.561776
{'data': {'ResponseBody': {'Date': '29-06-2026', 'ApiRemarks': None, 'ScheduleRemarks': 'FULLSCHEDULE_CREATED : From cron job', 'GroupWiseDataList': [{'ASList': [], 'Acronym': 'PVG_TATARENEWABLE', 'MTDLRefList': [], 'FullschdList': [{'ASTypeName': 'NA', 'FullScheduleData': {'ASFullScheduleJsonData': None, 'OAFullScheduleJsonData': {'SchdAmount': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.06, 1.82, 4.01, 6.7, 9.71, 12.91, 15.36, 17.62, 20.57, 23.64, 26.46, 28.65, 29.92, 30.14, 30.31, 31.59, 34.19, 36.56, 37.48, 36.91, 35.93, 35.3, 35.18, 35.45, 35.9, 36.18, 35.55, 34.01, 31.84, 30.41, 30.46, 31.65, 32.45, 32.25, 31.55, 30.9, 29.94, 28.6, 27.45, 26.79, 26.09, 24.41, 22.08, 19.58, 17.72, 15.96, 12.57, 9.51, 6.7, 4.34, 2.28, 0.47, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'POCDrawalLoss': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [6]:
schedule_data_row = row[3]

In [7]:
schedule_data_row

{'data': {'ResponseBody': {'Date': '29-06-2026',
   'ApiRemarks': None,
   'ScheduleRemarks': 'FULLSCHEDULE_CREATED : From cron job',
   'GroupWiseDataList': [{'ASList': [],
     'Acronym': 'PVG_TATARENEWABLE',
     'MTDLRefList': [],
     'FullschdList': [{'ASTypeName': 'NA',
       'FullScheduleData': {'ASFullScheduleJsonData': None,
        'OAFullScheduleJsonData': {'SchdAmount': [0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0,
          0.06,
          1.82,
          4.01,
          6.7,
          9.71,
          12.91,
          15.36,
          17.62,
          20.57,
          23.64,
          26.46,
          28.65,
          29.92,
          30.14,
          30.31,
          31.59,
          34.19,
          36.56,
          37.48,
      

In [ ]:
with open("latest_schedule_data.json", "w") as f:
    json.dump(schedule_data_row, f, indent=4) 

: 